# 3 · Supervised Fine-Tuning

The base model completes text. This notebook teaches it to answer. Nothing about the loop changes — same model, same loss, same optimizer — only the food: fifty thousand request→story pairs instead of a river of stories. About two minutes on a rented RTX 4090. The output is a run directory holding several finished chat models, each saved with its tokenizer in Hugging Face format. Chapter 4 turns one of them into a GGUF that `llama-cli` can talk to.

Two constants. The dataset is ours — the next few cells say why — and the context is the model's, fixed in chapter 2. Everything the notebook writes goes under `artifacts/`.

In [1]:
import json
from pathlib import Path

from datasets import load_dataset
from tokenizers import Tokenizer
from transformers import PreTrainedTokenizerFast

DATASET = "Pondsiders/tinystories-gpt4-instruct"
CONTEXT = 512
ARTIFACTS = Path("artifacts")

The reserved block from chapter 1 pays off. Training needs a padding token, and the tokenizer has five blanks; rename `<|reserved_3|>` to `<|pad|>` in the JSON and it exists, with no change to the model. `PreTrainedTokenizerFast` wraps the raw tokenizer with the roles Hugging Face's code asks about — which token begins text, which ends a turn, which is filler. End-of-sequence is `<|im_end|>` now, not `<|endoftext|>`: in a conversation, a turn ends and the text goes on.

In [2]:
raw = json.loads((ARTIFACTS / "tokenizer.json").read_text())

for entry in raw["added_tokens"]:
    if entry["content"] == "<|reserved_3|>":
        entry["content"] = "<|pad|>"
vocab = raw["model"]["vocab"]
if "<|reserved_3|>" in vocab:
    vocab["<|pad|>"] = vocab.pop("<|reserved_3|>")

tokenizer = PreTrainedTokenizerFast(
    tokenizer_object=Tokenizer.from_str(json.dumps(raw)),
    bos_token="<|endoftext|>",
    eos_token="<|im_end|>",
    pad_token="<|pad|>",
)

for name in ["<|endoftext|>", "<|im_start|>", "<|im_end|>", "<|pad|>"]:
    print(f"{tokenizer.convert_tokens_to_ids(name):4d}  {name}")

   0  <|endoftext|>
   1  <|im_start|>
   2  <|im_end|>
   3  <|pad|>


ChatML, the format most open chat models speak. A chat template is a Jinja string that turns a list of messages into one string, and this one is the whole grammar: `<|im_start|>`, a role, a newline, the content, `<|im_end|>`. The roles are ordinary text — `user` and `assistant` are tokenized like any words — so only two special tokens are spent. `add_generation_prompt` appends the opening of an assistant turn, which is exactly where the model will be asked to start writing.

In [3]:
CHAT_TEMPLATE = (
    "{{ '<|endoftext|>' }}"
    "{% for message in messages %}"
    "{{ '<|im_start|>' + message['role'] + '\n' + message['content'] + '<|im_end|>' + '\n' }}"
    "{% endfor %}"
    "{% if add_generation_prompt %}{{ '<|im_start|>assistant\n' }}{% endif %}"
)
tokenizer.chat_template = CHAT_TEMPLATE

messages = [{"role": "user", "content": "Tell me a story about a duck named Pondside."}]
print(tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=False))

<|endoftext|><|im_start|>user
Tell me a story about a duck named Pondside.<|im_end|>
<|im_start|>assistant



The data. Eldan and Li published TinyStories-Instruct, but its shape is `Words: / Features: / Summary:` — a specification, not a request. We wanted the thing a person types, so we made our own: 50,000 stories from the same corpus, each paired with a request that names what's in it, written from a dozen templates. Two templates were held out of training to test whether the model learns the *ask* or the *phrasing*. Every pair also carries `names`, `kind`, and `template_id`, so instruction-following can be graded by grep, with no judge model in the loop.

In [4]:
dataset = load_dataset(DATASET)
print(dataset)
print()
row = dataset["train"][0]
print(row["prompt"])
print(row["story"][:80] + "...")

README.md:   0%|          | 0.00/5.01k [00:00<?, ?B/s]

train.parquet: reconstructing file:   0%|          |  0.00B / 20.3MB            

train.parquet: downloading bytes:           |  0.00B            

validation.parquet: reconstructing file:   0%|          |  0.00B /  396kB            

validation.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/50000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['prompt', 'story', 'source_id', 'template_id', 'names', 'kind', 'all_names'],
        num_rows: 50000
    })
    validation: Dataset({
        features: ['prompt', 'story', 'source_id', 'template_id', 'names', 'kind', 'all_names'],
        num_rows: 1000
    })
})

I'd like a story about a boy named Tim, please.
One day, a little boy named Tim was eager to play outside. He saw that the sun w...


The loss mask needs one number the chat template never reports: the position where the response begins. Everything before it is context the model reads; everything from it on is what the model is graded on. So each pair is assembled as two lists and joined later — `request` ends with the assistant preamble, `response` begins with the first word of the story — and the boundary is the length of the first list. Assembling by hand is what makes that number exist.

In [5]:
BOS = tokenizer.convert_tokens_to_ids("<|endoftext|>")
IM_START = tokenizer.convert_tokens_to_ids("<|im_start|>")
IM_END = tokenizer.convert_tokens_to_ids("<|im_end|>")
PAD = tokenizer.convert_tokens_to_ids("<|pad|>")


def encode(text):
    return tokenizer.encode(text, add_special_tokens=False)


def assemble(prompt, story):
    request = (
        [BOS, IM_START] + encode("user\n" + prompt) + [IM_END]
        + encode("\n") + [IM_START] + encode("assistant\n")
    )
    response = encode(story) + [IM_END]
    return request, response


request, response = assemble(row["prompt"], row["story"])
print(tokenizer.decode(request), "…")
print(f"request {len(request)} tokens, response {len(response)} tokens")

<|endoftext|><|im_start|>user
I'd like a story about a boy named Tim, please.<|im_end|>
<|im_start|>assistant
 …
request 25 tokens, response 169 tokens


And the request half has to be the exact token sequence the chat template will produce at inference, because that's the string the model will actually be prompted with. Hand assembly and template rendering are two routes to the same tokens; if they disagree by so much as a newline, the model trains on one prompt and is asked with another, and nothing ever throws. This cell checks all thousand validation pairs. Zero mismatches is the number that lets the rest of the notebook proceed.

In [6]:
# Training must see exactly what inference will see: the request half of every
# training example has to match the chat template's rendering token for token.
mismatches = 0
for pair in dataset["validation"]:
    request, _ = assemble(pair["prompt"], pair["story"])
    templated = tokenizer.apply_chat_template(
        [{"role": "user", "content": pair["prompt"]}],
        add_generation_prompt=True,
    )["input_ids"]
    if request != templated:
        mismatches += 1
print(f"hand assembly vs chat template: {mismatches} mismatches in {len(dataset['validation'])} pairs")

hand assembly vs chat template: 0 mismatches in 1000 pairs


A story is a few hundred tokens; the context is 512. Sixty pairs in fifty thousand don't fit and get dropped rather than cut — a truncated story teaches the model to stop mid-sentence.

In [7]:
lengths = []
for pair in dataset["train"]:
    request, response = assemble(pair["prompt"], pair["story"])
    lengths.append(len(request) + len(response))

import numpy as np
lengths = np.array(lengths)
print(f"median {int(np.median(lengths))} tokens · p95 {int(np.percentile(lengths, 95))} · max {lengths.max()}")
print(f"over {CONTEXT}: {(lengths > CONTEXT).sum():,} of {len(lengths):,} ({100 * (lengths > CONTEXT).mean():.2f}%)")

median 204 tokens · p95 285 · max 1020
over 512: 60 of 50,000 (0.12%)


Pad, don't pack. Chapter 2 packed stories end to end because every token was a training target and waste mattered. Here only the story is graded, the request is context, and each example wants to start at position zero the way a real prompt will. So each pair becomes one row of 512: request, response, padding. Three arrays: `input_ids` is what the model reads, `labels` is what it's graded on — `-100` means *not this position* to PyTorch's loss — and `attention_mask` says which positions are real.

In [8]:
import numpy as np

IGNORE = -100


def build_example(pair):
    request, response = assemble(pair["prompt"], pair["story"])
    length = len(request) + len(response)
    if length > CONTEXT:
        return None
    pad = CONTEXT - length
    input_ids = request + response + [PAD] * pad
    labels = [IGNORE] * len(request) + response + [IGNORE] * pad
    attention_mask = [1] * length + [0] * pad
    return input_ids, labels, attention_mask


def build_split(split):
    examples = [build_example(pair) for pair in split]
    kept = [e for e in examples if e is not None]
    ids, labels, mask = (np.array(t, dtype=np.int16) for t in zip(*kept))
    print(f"{len(kept):,} examples kept, {len(examples) - len(kept):,} dropped for length")
    return ids, labels, mask


train_ids, train_labels, train_mask = build_split(dataset["train"])
valid_ids, valid_labels, valid_mask = build_split(dataset["validation"])

49,940 examples kept, 60 dropped for length
997 examples kept, 3 dropped for length


The pad token and `-100` wear different hats and must agree. If padding leaks into the loss, `<|pad|>` becomes the most common next token in every long sequence and the model learns to predict nothing, while the loss looks wonderful. The assertion is cheap and the bug it catches is silent. And the graded region ends with `<|im_end|>` on purpose: that token is how the model learns that stories end.

In [9]:
# The pad token and -100 wear different hats but must agree: anywhere the
# input is padding, the label must be IGNORE, or the model learns to predict
# nothing and the loss looks wonderful while the model gets worse.
assert (train_labels[train_ids == PAD] == IGNORE).all()
assert (valid_labels[valid_ids == PAD] == IGNORE).all()

# And the graded region must be exactly the response: story plus its <|im_end|>.
graded = train_labels[0][train_labels[0] != IGNORE].astype(np.int64)
print(tokenizer.decode(graded)[:120] + " …")
print(f"…ends with: {tokenizer.convert_ids_to_tokens([int(graded[-1])])}")

One day, a little boy named Tim was eager to play outside. He saw that the sun was shining and the snow was melting. He  …
…ends with: ['<|im_end|>']


One example, drawn. Dots are ungraded, blocks are graded. The whole request — the user's words and the `assistant` preamble — is context the model reads but isn't scored on. The story is the test.

In [10]:
example = train_ids[0].astype(np.int64)
markers = ["·" if label == IGNORE else "█" for label in train_labels[0]]
tokens = tokenizer.convert_ids_to_tokens(example)
print("".join(markers[:80]))
print()
for token, marker in list(zip(tokens, markers))[:24]:
    print(f"  {marker}  {token}")

·························███████████████████████████████████████████████████████

  ·  <|endoftext|>
  ·  <|im_start|>
  ·  us
  ·  er
  ·  Ċ
  ·  I
  ·  'd
  ·  Ġlike
  ·  Ġa
  ·  Ġstory
  ·  Ġabout
  ·  Ġa
  ·  Ġboy
  ·  Ġnamed
  ·  ĠTim
  ·  ,
  ·  Ġplease
  ·  .
  ·  <|im_end|>
  ·  Ċ
  ·  <|im_start|>
  ·  ass
  ·  ist
  ·  ant


Ten million real tokens, nine million of them graded — about two percent of what pretraining saw. And 59% of all positions are padding, which the batching below fixes: rows are stored at 512 wide, but a batch only needs to be as wide as its longest member.

In [11]:
total = train_mask.sum()
graded = (train_labels != IGNORE).sum()
padding = (train_ids == PAD).sum()
print(f"tokens in play: {total:,}")
print(f"graded: {graded:,} ({100 * graded / total:.0f}% of real tokens)")
print(f"padding: {padding:,} ({100 * padding / train_ids.size:.0f}% of all positions)")

tokens in play: 10,493,205
graded: 9,367,757 (89% of real tokens)
padding: 15,076,075 (59% of all positions)


Hyperparameters, and one of them carries the chapter. Peak learning rate is `6e-5` — ten times lower than pretraining's `6e-4`. The base already knows English; we're adjusting a model, not building one, and a big step here would erase what took nine minutes to learn. Precision matches chapter 2: fp32 weights, because the optimizer needs the resolution to feel small updates, and bf16 for the forward and backward passes. Every run gets a directory named for when it started and a `config.json` saying what it was and what it ran on, so two runs on two GPUs can't overwrite each other and can be told apart afterward. Milestones are counted in pairs seen; the last two are one epoch and three.

In [12]:
import json
from contextlib import nullcontext

import pendulum
import torch
import transformers
from transformers import LlamaForCausalLM

BATCH_SIZE = 64
PEAK_LR = 6e-5
WARMUP_STEPS = 50
TRUNK_EPOCHS = 3
ANNEAL_STEPS = 100
GRAD_CLIP = 1.0
SEED = 20260831

STEPS_PER_EPOCH = len(train_ids) // BATCH_SIZE
TRUNK_STEPS = TRUNK_EPOCHS * STEPS_PER_EPOCH
PAIRS_PER_EPOCH = STEPS_PER_EPOCH * BATCH_SIZE

# Checkpoints peel off the trunk when this many pairs have been seen. The last
# two are one full epoch and the end of the trunk.
MILESTONES = [1_000, 5_000, 20_000, PAIRS_PER_EPOCH, TRUNK_EPOCHS * PAIRS_PER_EPOCH]

device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
torch.manual_seed(SEED)

# Mixed precision, as in pretraining: the weights stay fp32 (the optimizer
# needs the precision to feel small updates), the forward and backward passes
# run in bf16.
amp = nullcontext() if device == "cpu" else torch.autocast(device, dtype=torch.bfloat16)

model = LlamaForCausalLM.from_pretrained(ARTIFACTS / "lil-transformy-2").to(device)
model.train()

optimizer = torch.optim.AdamW(
    model.parameters(), lr=PEAK_LR, betas=(0.9, 0.95), weight_decay=0.1,
    fused=(device == "cuda"),
)

# Every run gets its own directory, named by when it started (house time), and
# a config.json saying what it was and what it ran on.
RUN = ARTIFACTS / "sft" / pendulum.now("America/Los_Angeles").format("YYYY-MM-DD_HHmm")
RUN.mkdir(parents=True, exist_ok=True)
config = {
    "run": RUN.name,
    "started": pendulum.now("America/Los_Angeles").to_datetime_string(),
    "device": device,
    "gpu": torch.cuda.get_device_name(0) if device == "cuda" else None,
    "torch": torch.__version__,
    "transformers": transformers.__version__,
    "base_model": str(ARTIFACTS / "lil-transformy-2"),
    "dataset": DATASET,
    "context": CONTEXT,
    "precision": "bf16 autocast over fp32 masters",
    "batch_size": BATCH_SIZE,
    "peak_lr": PEAK_LR,
    "warmup_steps": WARMUP_STEPS,
    "trunk_epochs": TRUNK_EPOCHS,
    "trunk_steps": TRUNK_STEPS,
    "anneal_steps": ANNEAL_STEPS,
    "grad_clip": GRAD_CLIP,
    "seed": SEED,
    "milestones": MILESTONES,
}
(RUN / "config.json").write_text(json.dumps(config, indent=2) + "\n")

print(f"run {RUN}")
print(f"{config['gpu'] or device} · {STEPS_PER_EPOCH} steps/epoch · trunk {TRUNK_STEPS} steps · "
      f"milestones at pairs {MILESTONES}")

Loading weights:   0%|          | 0/74 [00:00<?, ?it/s]

run artifacts/sft/2026-09-05_0958
NVIDIA GeForce RTX 4090 · 780 steps/epoch · trunk 2340 steps · milestones at pairs [1000, 5000, 20000, 49920, 149760]


Warmup-Stable-Decay. Pretraining cooled its learning rate down along a cosine, which commits you to the run's end before it starts. Here the trunk warms up and then holds flat, and every checkpoint peeled off it gets its own short cooldown — so one run can father many finished models, and you choose among them afterward. `to_batch` trims each batch to its longest example before it goes to the GPU. `train_step` is shared by the trunk and the anneals; as in chapter 2, only `opt.step()` changes the model.

In [13]:
# Warmup-Stable-Decay: warm up, then hold the trunk FLAT. There is no decay
# here — every checkpoint peeled off the trunk gets its own short anneal
# instead, so one run can father many finished models.
def trunk_lr(step):
    if step < WARMUP_STEPS:
        return PEAK_LR * (step + 1) / WARMUP_STEPS
    return PEAK_LR


# Pad to the batch, not to the context. Every example was padded out to 512
# positions on disk; a batch only needs to be as wide as its longest member.
def to_batch(ids, mask, labels):
    width = int(mask.sum(axis=1).max())
    return {
        "input_ids": torch.from_numpy(ids[:, :width].astype(np.int64)).to(device),
        "attention_mask": torch.from_numpy(mask[:, :width].astype(np.int64)).to(device),
        "labels": torch.from_numpy(labels[:, :width].astype(np.int64)).to(device),
    }


def batches(epoch_seed):
    order = np.random.default_rng(epoch_seed).permutation(len(train_ids))
    for start in range(0, len(order) - BATCH_SIZE + 1, BATCH_SIZE):
        rows = order[start : start + BATCH_SIZE]
        yield to_batch(train_ids[rows], train_mask[rows], train_labels[rows])


def train_step(net, opt, batch, lr):
    for group in opt.param_groups:
        group["lr"] = lr
    with amp:
        loss = net(**batch).loss
    opt.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(net.parameters(), GRAD_CLIP)
    opt.step()
    return loss


@torch.no_grad()
def validation_loss(net, n_batches=8):
    net.eval()
    total = 0.0
    for start in range(0, n_batches * BATCH_SIZE, BATCH_SIZE):
        rows = slice(start, start + BATCH_SIZE)
        with amp:
            total += net(**to_batch(valid_ids[rows], valid_mask[rows], valid_labels[rows])).loss.item()
    net.train()
    return total / n_batches

Read the numbers. Before a single step the base scores 1.40 on this data, a tenth of a nat above its pretraining loss — it looked at a chat template and saw some junk tokens followed by a story, which it knows how to write. Sixteen steps later, 1.21. The bottom is 1.0867 at the end of the first epoch, and then validation climbs, slowly and monotonically, to 1.0947 by the end of the third. That climb is overfitting: eight million parameters can't memorize fifty thousand stories, but on the second look they can start to prefer these fifty thousand over the thousand they're graded on. Three epochs was this notebook's default and the run corrected it. The remedy is called early stopping, and this loop does the reliable version of it: train through, keep checkpoints, pick the best afterward. The optimizer state is saved beside each checkpoint so its cooldown continues the trunk instead of restarting it.

In [14]:
# The trunk. Three passes over the pairs at a flat learning rate. When the
# count of pairs seen crosses a milestone, the fp32 weights and the optimizer
# state are saved as they are — no cooldown yet, that comes next.
started = pendulum.now()
print(f"validation loss before training: {validation_loss(model):.4f}")

peeled = []
pending = list(MILESTONES)
step = 0
pairs_seen = 0
history = []

for epoch in range(TRUNK_EPOCHS):
    for batch in batches(epoch_seed=SEED + epoch):
        loss = train_step(model, optimizer, batch, trunk_lr(step))
        step += 1
        pairs_seen += BATCH_SIZE

        if step % 100 == 0 or step == TRUNK_STEPS:
            valid = validation_loss(model)
            history.append((step, pairs_seen, loss.item(), valid))
            print(f"step {step:5d} · pairs {pairs_seen:7,d} · lr {trunk_lr(step - 1):.1e} · "
                  f"train {loss.item():.4f} · valid {valid:.4f}")

        if pending and pairs_seen >= pending[0]:
            pending.pop(0)
            where = RUN / "trunk" / f"pairs-{pairs_seen:06d}"
            model.save_pretrained(where)
            torch.save(optimizer.state_dict(), where / "optimizer.pt")
            peeled.append(where)
            print(f"  peeled {where.relative_to(RUN)} · valid {validation_loss(model):.4f}")

elapsed = pendulum.now() - started
config["trunk_seconds"] = round(elapsed.total_seconds(), 1)
(RUN / "config.json").write_text(json.dumps(config, indent=2) + "\n")
print(f"trunk done in {elapsed.in_words()} · {TRUNK_STEPS / elapsed.total_seconds():.1f} steps/s")

validation loss before training: 1.4018


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  peeled trunk/pairs-001024 · valid 1.2083


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  peeled trunk/pairs-005056 · valid 1.1117
step   100 · pairs   6,400 · lr 6.0e-05 · train 1.1598 · valid 1.1047
step   200 · pairs  12,800 · lr 6.0e-05 · train 1.1405 · valid 1.0908
step   300 · pairs  19,200 · lr 6.0e-05 · train 1.2550 · valid 1.0901


...

  peeled trunk/pairs-020032 · valid 1.0896
step   400 · pairs  25,600 · lr 6.0e-05 · train 1.2662 · valid 1.0884
step   500 · pairs  32,000 · lr 6.0e-05 · train 1.2384 · valid 1.0880
step   600 · pairs  38,400 · lr 6.0e-05 · train 1.1155 · valid 1.0882
step   700 · pairs  44,800 · lr 6.0e-05 · train 1.1979 · valid 1.0875


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  peeled trunk/pairs-049920 · valid 1.0867
step   800 · pairs  51,200 · lr 6.0e-05 · train 1.0809 · valid 1.0881
step   900 · pairs  57,600 · lr 6.0e-05 · train 1.1709 · valid 1.0910
step  1000 · pairs  64,000 · lr 6.0e-05 · train 1.1934 · valid 1.0901
step  1100 · pairs  70,400 · lr 6.0e-05 · train 1.2321 · valid 1.0903
step  1200 · pairs  76,800 · lr 6.0e-05 · train 1.1113 · valid 1.0912
step  1300 · pairs  83,200 · lr 6.0e-05 · train 1.1787 · valid 1.0902
step  1400 · pairs  89,600 · lr 6.0e-05 · train 1.1267 · valid 1.0902
step  1500 · pairs  96,000 · lr 6.0e-05 · train 1.2168 · valid 1.0907
step  1600 · pairs 102,400 · lr 6.0e-05 · train 1.0393 · valid 1.0925
step  1700 · pairs 108,800 · lr 6.0e-05 · train 1.1013 · valid 1.0935
step  1800 · pairs 115,200 · lr 6.0e-05 · train 1.1741 · valid 1.0955
step  1900 · pairs 121,600 · lr 6.0e-05 · train 1.1176 · valid 1.0950
step  2000 · pairs 128,000 · lr 6.0e-05 · train 1.1041 · valid 1.0945
step  2100 · pairs 134,400 · lr 6.0e-05 · train

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  peeled trunk/pairs-149760 · valid 1.0947
trunk done in 2 minutes 1 second · 19.3 steps/s


Each peeled checkpoint runs a hundred steps with the learning rate falling linearly to zero, then saves in bf16 with the tokenizer and chat template riding along — nothing will train from these, so they can be small. The cooldown matters most where the model is least settled: the 1K checkpoint gains a tenth of a nat from its anneal, the final one almost nothing. The best model of the run is `pairs-049920`, one epoch, at 1.0837.

The 1K checkpoint is also worth a sentence. Under sampling it writes a complete story, then dissolves into bytes like `Ĳ` and `÷` before starting a fresh story. `<|im_end|>` had never been a training target before this notebook, so it began life in the same corner of the embedding table as every other never-seen token — two hundred rows pointing the same way. At step 16 the model already puts most of its probability on that corner; it just can't say which row it means. By 20K pairs the row has pulled away from its neighbors and the model stops cleanly.

In [15]:
# The anneals. Each peeled checkpoint picks up exactly where the trunk left
# it — same weights, same optimizer moments — and runs the learning rate
# linearly down to zero. The result is saved in bf16 with the tokenizer and
# chat template riding along: a complete model, nothing will train from it.
def anneal(trunk_dir, seed):
    net = LlamaForCausalLM.from_pretrained(trunk_dir).to(device)
    net.train()
    opt = torch.optim.AdamW(
        net.parameters(), lr=PEAK_LR, betas=(0.9, 0.95), weight_decay=0.1,
        fused=(device == "cuda"),
    )
    opt.load_state_dict(torch.load(trunk_dir / "optimizer.pt", map_location=device))
    for i, batch in enumerate(batches(epoch_seed=seed)):
        if i >= ANNEAL_STEPS:
            break
        train_step(net, opt, batch, PEAK_LR * (ANNEAL_STEPS - i) / ANNEAL_STEPS)
    return net


for i, trunk_dir in enumerate(peeled):
    net = anneal(trunk_dir, seed=SEED + 1_000 + i)
    valid = validation_loss(net)
    out = RUN / "annealed" / trunk_dir.name
    net.to(torch.bfloat16).save_pretrained(out)
    tokenizer.save_pretrained(out)
    print(f"{out.relative_to(RUN)} · valid {valid:.4f}")

Loading weights:   0%|          | 0/74 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

annealed/pairs-001024 · valid 1.1079


Loading weights:   0%|          | 0/74 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

annealed/pairs-049920 · valid 1.0837


Loading weights:   0%|          | 0/74 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

annealed/pairs-149760 · valid 1.0933


A request no template used, with a constraint the data never posed. The story is about a duck, it's named Pond-*ath* — `Pondside` is three tokens, `P`, `ond`, `side`, and the model glued a name-shaped ending onto the first two — and it stops by itself. The shiny stone never appears. One constraint of two, from eight million parameters and two minutes of training. Sampling is the house recipe from chapter 2. This cell loads the final checkpoint; by the numbers above, the one-epoch model would have been the better choice, and the next chapter quantizes that one.

In [22]:
# The moment of truth: a request the model has never seen, in words no
# template used. House recipe — temperature 0.7, top_p 0.95.
final = LlamaForCausalLM.from_pretrained(RUN / "annealed" / peeled[-1].name).to(device)
final.eval()

request = "Could you tell me a story about a duck named Pondside who finds a shiny stone?"
prompt = tokenizer.apply_chat_template(
    [{"role": "user", "content": request}], add_generation_prompt=True, return_tensors="pt",
)["input_ids"].to(device)

torch.manual_seed(SEED)
with torch.no_grad():
    out = final.generate(
        prompt, max_new_tokens=300, do_sample=True, temperature=0.7, top_p=0.95,
        eos_token_id=IM_END, pad_token_id=PAD,
    )
story = tokenizer.decode(out[0][prompt.shape[1]:], skip_special_tokens=True)
print(request)
print()
print(story)
print()
print(f"run {RUN}")

Loading weights:   0%|          | 0/74 [00:00<?, ?it/s]

Could you tell me a story about a duck named Pondside who finds a shiny stone?

Once upon a time, there was a little duck named Pondath. Pondath loved to splash in the pond all day long. One day, Pondath was so excited to see the beautiful water that he didn't want to stop splashing.
Pondath's mum saw him and said, "Don't splash too much, Pondath! It's not safe to splash so much!"
Pondath felt a little bit ashamed but she didn't want to take the time to splash. So, she said, "Okay, mum. I won't splash so much."
Mum smiled and said, "Let's go to the store and buy a special treat for us. What do you want?"
Pondath thought for a moment and then said, "I want a yummy treat!"
Mum said, "I know! You can have a sweet treat if you get a tip first!"
Pondath was so excited and couldn't wait to try the new treat. The two of them went to the store and bought a yummy treat for Pondath.
When they got home, Pondath put on her special treat and started to splash around. She was so happy that she hadn'